In [1]:
#mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
import json
import os


# Define your project root path (adjust this if you named your folder differently)
PROJECT_ROOT = '/content/drive/MyDrive/Advanced Machine Learning Project'

# Add the project root to the system path so Python can find the 'src' folder
sys.path.append(PROJECT_ROOT)

print("Setup complete. Project root added to path.")

Setup complete. Project root added to path.


In [14]:
import pandas as pd
import json
import os

# Load the config file (assuming PROJECT_ROOT is already defined)
config_path = os.path.join(PROJECT_ROOT, 'config.json')
with open(config_path, 'r') as f:
    config = json.load(f)

# Load Nasdaq 100 tickers from local CSV file
print("Loading Nasdaq tickers from local CSV...")
tickers_file_path = os.path.join(PROJECT_ROOT, 'data', 'nasdaq_100_ordered.csv')
df = pd.read_csv(tickers_file_path)
tickers = df['Ticker'].tolist()

# Replace dots with hyphens for yfinance (e.g., BRK.B -> BRK-B)
tickers = [t.replace('.', '-') for t in tickers]

# Pull variables directly from your config
top_n = config['data']['top_n_tickers']
target_ticker = config['data']['target_ticker']

# Keep only the top N tickers based on the config
top_tickers = tickers[:top_n]

print("Top tickers loaded from config:")
for ticker in top_tickers:
    print(ticker)

# Ensure our target ticker is in the list
if target_ticker not in top_tickers:
    top_tickers.append(target_ticker)

print(f"\nTracking {len(top_tickers)} tickers. Target is: {target_ticker}")

Loading Nasdaq tickers from local CSV...
Top tickers loaded from config:
NVDA
AAPL
MSFT
AMZN
GOOGL
GOOG
AVGO
META
TSLA
WMT
ASML
MU
AMD
COST
NFLX
PLTR
CSCO
INTC
LRCX
AMAT

Tracking 20 tickers. Target is: NVDA


In [15]:
# Cell 3: Execute Data Pipeline
from src.data_loader import download_and_prepare_data

start_date = config['data']['start_date']
end_date = config['data']['end_date']

# Download and clean the data
log_returns, binary_target = download_and_prepare_data(
    tickers=top_tickers,
    start_date=start_date,
    end_date=end_date,
    target_ticker=target_ticker
)

# Optional: Save the raw pandas dataframes just in case you want to look at them later
data_folder = os.path.join(PROJECT_ROOT, 'data')
os.makedirs(data_folder, exist_ok=True)

log_returns.to_csv(os.path.join(data_folder, 'log_returns.csv'))
binary_target.to_csv(os.path.join(data_folder, 'binary_targets.csv'))

print("Dataframes saved to CSV successfully!")

/content/drive/MyDrive/Advanced Machine Learning Project/src/data_loader.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start=start_date, end=end_date)['Close']
[************          25%                       ]  5 of 20 completed

[*********************100%***********************]  20 of 20 completed


Data prepared. Shape: (2262, 19)
Dataframes saved to CSV successfully!


In [16]:
# Cell 4: Create PyTorch Dataset and Save Tensors
import torch
from src.dataset import FinancialSequenceDataset

sequence_length = config['data']['sequence_length']

# Instantiate the dataset
dataset = FinancialSequenceDataset(log_returns, binary_target, sequence_length)

# Extract all X and y pairs from the dataset
# We do this so we can save the raw tensors directly to disk for fast loading later
all_X = []
all_y = []

print("Building sliding windows...")
for i in range(len(dataset)):
    x, y = dataset[i]
    all_X.append(x)
    all_y.append(y)

# Stack them into massive single tensors
X_tensor = torch.stack(all_X)
y_tensor = torch.stack(all_y)

print(f"Final Input Tensor Shape: {X_tensor.shape}")
print(f"Final Target Tensor Shape: {y_tensor.shape}")

# Save the PyTorch tensors
tensor_path = os.path.join(data_folder, 'processed_tensors.pt')
torch.save({'X': X_tensor, 'y': y_tensor}, tensor_path)

print(f"Tensors successfully saved to: {tensor_path}")


Building sliding windows...
Final Input Tensor Shape: torch.Size([2232, 30, 19])
Final Target Tensor Shape: torch.Size([2232])
Tensors successfully saved to: /content/drive/MyDrive/Advanced Machine Learning Project/data/processed_tensors.pt
